# ViSceT5 — Pretrain **gen_all** (decoder read-scene-text, đòn bẩy #1)
Chạy tuần tự. `gen_all` = huấn luyện decoder **sinh scene-text** (khớp đúng đường finetune: encoder chỉ nhận câu hỏi + ảnh + OCR-feature) + MLM/ITM/TWC làm phụ trợ (×0.5) — phần pretrain trực tiếp có ích cho bộ sinh câu trả lời seq2seq.

Sau khi pretrain xong & upload lên HF, dùng `notebooks/finetune_colab.ipynb` để finetune từ nó.

In [ ]:
!git clone https://github.com/Kussssssss/ViSceT5.git
%cd ViSceT5
!git pull

In [ ]:
!bash setup.sh
# Nếu Colab báo cần restart: Runtime > Restart, rồi chạy tiếp TỪ cell cấu hình.

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_xxx'                       # <== ĐIỀN token HF của bạn
HF_PRETRAIN_REPO = 'Kus669/ViSceT5-pretrain-genall'     # repo sẽ lưu MODEL PRETRAIN (gen_all)

In [ ]:
import argparse
from scripts import prepare_dataset
prepare_dataset.main(argparse.Namespace(config='configs/data/ViTextVQA.yaml', data_dir='./datasets'))

In [ ]:
from scripts import init_model
init_model.main()

### 1) SMOKE / MOCK — in debug ĐẦY ĐỦ để đảm bảo các phần chạy đúng
Bật `TWC_TRAIN_LOG=1` để in debug per-step. Trong log hãy tìm đủ 5 mục:
1. `🧊➡️🔥 [pretrain] vision unfreeze: last 2 ...` → **vision unfreeze** OK
2. `>>> [pretrain] MLM mask mode = wholeword` → **whole-word masking** đang bật
3. Block `🔬 [VERIFY]` có `✅ [GEN] gen_loss finite & > 0` + `✅ [GEN] gen_loss requires grad` → **gen** OK
4. Block `🔍 [DIAG] MLM 'copy-crutch' A/B` → so **wholeword vs subword**: `acc FULL` vs `acc OCR-ablated` + `DROP` + `mask-targets(sample0)` (thấy nguyên từ bị mask)
5. Per-step `[Pretrain] ... Loss(M):.. Loss(I):.. Loss(TWC):.. Loss(GEN):..` → cả **4 loss** đều chạy

Phải truyền `args_list=[...]` (không dùng sys.argv) để mode có tác dụng trong kernel.

In [ ]:
import os, importlib
os.environ['TWC_TRAIN_LOG'] = '1'   # in debug per-step (Loss M/I/TWC/GEN) trong vòng lặp mock
from training import pretrain
importlib.reload(pretrain)
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--vision_unfreeze_last_n', '2',   # mở băng 2 lớp vision cuối (học đặc trưng thị giác)
    '--mlm_mask_mode', 'wholeword',    # mask trọn từ (bỏ 'nạng' copy subword)
    '--smoke_test', 'True',
])

### 2) FULL PRETRAIN gen_all (+ mở băng vision)
`--vision_unfreeze_last_n 2` mở băng 2 lớp CLIP-vision cuối để pretrain **học đặc trưng thị giác** (đúng bản chất transfer learning); tìm log `🧊➡️🔥 [QACLIP] unfroze last 2 ... vision layers`. Đặt `0` để giữ backbone đóng băng. Theo dõi `loss_gen` trong log eval (giảm dần).

In [ ]:
import importlib
from training import pretrain
importlib.reload(pretrain)
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--vision_unfreeze_last_n', '2',   # 0 = đóng băng vision; 2-4 = học đặc trưng thị giác
    '--num_train_epochs', '3',
])

### 3) Upload model pretrain lên HF (để finetune_colab.ipynb dùng)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(repo_id=HF_PRETRAIN_REPO, repo_type='model', exist_ok=True)
api.upload_folder(folder_path='/content/ViSceT5/output/pretrain', repo_id=HF_PRETRAIN_REPO,
                  repo_type='model', ignore_patterns=['optimizer.pt'])
print('Uploaded pretrain ->', HF_PRETRAIN_REPO)